# Standardize HoleNo across a web map

Brings an older staking/as-built web map up to the current template:

1. **Schema** - adds a `HoleNo` text field (with the standard hole-number dropdown) to every template layer that doesn't have one, and updates the dropdown on layers that already have it.
2. **Existing data** - remaps old values to the new codes (e.g. `Clubhouse` -> `CH`).
3. **Field Maps forms** - adds a HoleNo dropdown to each layer's form in the web map (just above Collection Date, like the new template) so crews can fill it in.

**Run with `DRY_RUN = True` first** - it only prints what it would change. Then set `DRY_RUN = False` and re-run all cells to apply. A backup of the web map JSON is written before anything is saved.

Works in ArcGIS Online Notebooks or in ArcGIS Pro (uses your Pro sign-in).

In [ ]:
# ---------------- PARAMETERS ----------------
WEBMAP_ID = "0b71d91d79d146a98775d056d9492779"   # 2603 Kettle Forge Staking
DRY_RUN = True

FIELD_NAME = "HoleNo"
# Standard hole list (matches the newer templates, plus CG). Order = order in the dropdown.
HOLE_CODES = [f"{i:02d}" for i in range(1, 19)] + ["CH", "SG", "PG", "CG", "DR"]
# Old value -> new code, applied to features that already have data
VALUE_REMAP = {"Clubhouse": "CH"}

# Only layers published from the template are touched. These are detected automatically
# (services sharing the template's 32-character name suffix). Force a layer in or out by its map title:
EXTRA_INCLUDE_TITLES = []
EXCLUDE_TITLES = []

In [ ]:
import json, re, datetime, collections
from arcgis.gis import GIS
from arcgis.features import FeatureLayer

try:
    gis = GIS("home")      # ArcGIS Online notebook
except Exception:
    gis = GIS("pro")       # ArcGIS Pro / local Python
print("Signed in as", gis.users.me.username)

wm_item = gis.content.get(WEBMAP_ID)
wm_data = wm_item.get_data()
print("Web map:", wm_item.title)

def iter_layers(layers):
    for l in layers:
        yield l
        yield from iter_layers(l.get("layers", []))

feature_layers = [l for l in iter_layers(wm_data["operationalLayers"])
                  if l.get("layerType") == "ArcGISFeatureLayer" and l.get("url")]

SUFFIX_RE = re.compile(r"_([0-9a-f]{32})/FeatureServer", re.I)
suffixes = collections.Counter(m.group(1) for l in feature_layers if (m := SUFFIX_RE.search(l["url"])))
template_suffix = suffixes.most_common(1)[0][0] if suffixes else None
print("Template service suffix:", template_suffix)

def is_target(l):
    t = l.get("title")
    if t in EXCLUDE_TITLES: return False
    if t in EXTRA_INCLUDE_TITLES: return True
    m = SUFFIX_RE.search(l["url"])
    return bool(m and m.group(1) == template_suffix)

targets = [l for l in feature_layers if is_target(l)]
skipped = [l["title"] for l in feature_layers if not is_target(l)]
print(f"{len(targets)} template layers to process; skipping {len(skipped)}: {skipped}")

In [ ]:
DOMAIN = {"type": "codedValue", "name": f"{FIELD_NAME}_Standard",
          "codedValues": [{"name": c, "code": c} for c in HOLE_CODES]}
NEW_FIELD = {"name": FIELD_NAME, "type": "esriFieldTypeString", "alias": FIELD_NAME, "length": 256,
             "nullable": True, "editable": True, "domain": DOMAIN, "defaultValue": None}
FORM_ELEMENT = {"label": FIELD_NAME, "type": "field", "editableExpression": "expr/system/true",
                "fieldName": FIELD_NAME,
                "inputType": {"type": "combo-box", "noValueOptionLabel": "No value", "showNoValueOption": True}}

def find_field(fl):
    for f in fl.properties.fields:
        if f["name"].lower() == FIELD_NAME.lower():
            return f
    return None

def form_has_field(form_elements, name):
    for e in form_elements:
        if e.get("type") == "group" and form_has_field(e.get("formElements", []), name): return True
        if e.get("fieldName", "").lower() == name.lower(): return True
    return False

def insert_form_element(form_info, element):
    """Insert into the first group, just above Collection Date (or at the end of the group)."""
    container = form_info["formElements"]
    groups = [e for e in container if e.get("type") == "group"]
    if groups: container = groups[0]["formElements"]
    idx = next((i for i, e in enumerate(container) if e.get("fieldName") == "Collection_Date"), len(container))
    container.insert(idx, dict(element))

plan = []   # (layer_title, FeatureLayer, actions)
for l in targets:
    fl = FeatureLayer(l["url"], gis)
    actions = []
    field = find_field(fl)
    if field is None:
        actions.append("ADD FIELD")
    else:
        current = [c["code"] for c in (field.get("domain") or {}).get("codedValues", [])]
        if current != HOLE_CODES:
            actions.append(f"UPDATE DROPDOWN (was {current or 'no dropdown'})")
        if VALUE_REMAP:
            where = " OR ".join(f"{field['name']} = '{old}'" for old in VALUE_REMAP)
            n = fl.query(where=where, return_count_only=True)
            if n: actions.append(f"REMAP VALUES on {n} features")
    if l.get("formInfo") and not form_has_field(l["formInfo"]["formElements"], FIELD_NAME):
        actions.append("ADD TO FORM")
    plan.append((l["title"], fl, actions))

for title, _, actions in plan:
    print(f"{title:32s} {', '.join(actions) if actions else 'ok - nothing to do'}")
print(f"\n{sum(1 for p in plan if p[2])} of {len(plan)} layers need changes.  DRY_RUN = {DRY_RUN}")

In [ ]:
if DRY_RUN:
    print("Dry run only - nothing changed. Set DRY_RUN = False and re-run all cells to apply.")
else:
    # 1) Backup the web map JSON before touching anything
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = f"webmap_backup_{WEBMAP_ID}_{stamp}.json"
    with open(backup_path, "w", encoding="utf-8") as f:
        json.dump(wm_data, f, indent=1)
    print("Web map backup written to", backup_path)

    errors = []
    for title, fl, actions in plan:
        try:
            if "ADD FIELD" in actions:
                r = fl.manager.add_to_definition({"fields": [NEW_FIELD]})
                print(f"{title}: add field -> {r}")
            elif any(a.startswith("UPDATE DROPDOWN") for a in actions):
                existing = find_field(fl)
                r = fl.manager.update_definition({"fields": [{"name": existing["name"], "domain": DOMAIN}]})
                print(f"{title}: update dropdown -> {r}")
            if any(a.startswith("REMAP VALUES") for a in actions):
                fname = find_field(fl)["name"]
                for old, new in VALUE_REMAP.items():
                    r = fl.calculate(where=f"{fname} = '{old}'", calc_expression={"field": fname, "value": new})
                    print(f"{title}: remap {old} -> {new}: {r}")
        except Exception as e:
            errors.append((title, str(e)))
            print(f"!! {title}: {e}")

    # 2) Field Maps forms (stored in the web map item)
    form_changes = 0
    for l in targets:
        if l.get("formInfo") and not form_has_field(l["formInfo"]["formElements"], FIELD_NAME):
            insert_form_element(l["formInfo"], FORM_ELEMENT)
            form_changes += 1
    if form_changes:
        ok = wm_item.update(data=json.dumps(wm_data))
        print(f"Added HoleNo to {form_changes} forms; web map saved: {ok}")

    print("\nDone." if not errors else f"\nFinished with {len(errors)} errors: {errors}")